# Firm characteristics: the same factor structure with a nonlinear map

[`08a_ipca`](08a_ipca.ipynb) required a firm's exposure to each latent factor to be a **linear**
function of its characteristics that month. That constraint is what let one map be estimated on
the whole panel instead of 57 weights on one cross-section, and it is also the assumption most
likely to be wrong: nothing says the relationship between a firm's book-to-market rank and its
exposure to a value factor is a straight line, and a product of two characteristics - which
`03_financial_features` builds explicitly for exactly this reason - is not linear in either.

The **conditional autoencoder** keeps everything else and replaces only that map. A network takes
the firm's characteristics and returns its exposures; a second, linear part of the model turns
the month's cross-section of returns into that month's factor returns; the predicted return is
the product, as it was in IPCA. So the difference between this notebook and the last one is the
shape of one function, which is what makes them worth reading against each other.

**This one has a stopping point and the last one did not.** IPCA runs alternating least squares
to convergence, so a fold produces one fitted map. A network is trained for a declared number of
epochs, and its state after 5 epochs is a different model from its state after 50 - so
`case_studies/config/cae/cae.yaml` declares both the budget and how often to save, and every
saved state becomes its own registered prediction set. The checkpoint is part of the
configuration, not a detail of how it was fitted.

**Learning objectives**


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 公司特徵：保留相同因子結構，改用非線性映射

[`08a_ipca`](08a_ipca.ipynb) 要求公司對各潛在因子的曝險，是其當月特徵的**線性**函數。這項限制使模型得以在整份縱橫資料上估計同一個映射，而非在單月橫斷面估計 57 個權重；但它也最可能與現實不符：沒有理由認為公司的帳面市值比排名與價值因子曝險之間必然是直線關係，而兩項特徵的乘積——`03_financial_features` 正因如此明確建立它——對任一特徵都不是線性函數。

**條件式自編碼器**保留其他結構，只取代該映射。網路讀取公司的特徵並回傳其曝險；模型的第二個線性部分將當月報酬橫斷面轉換為該月的因子報酬；預測報酬如 IPCA 一樣是兩者的乘積。因此，本 Notebook 與上一份 Notebook 只在一個函數的形狀上不同，值得相互對照。

**此模型有停止時點，上一個沒有。** IPCA 交替執行最小平方法直到收斂，因此每折只產生一個擬合映射。網路依宣告的訓練週期數訓練，訓練 5 個週期後的狀態與 50 個週期後不同；因此 `case_studies/config/cae/cae.yaml` 同時宣告訓練預算與儲存頻率，每個已儲存狀態都成為獨立登錄的預測集合。檢查點是設定的一部分，而非擬合過程的細節。

**學習目標**
<!-- ML4T_ZH_END c0000-p000 -->

- Separate what a latent-factor model assumes about structure from what it assumes about
  functional form.
- Read a checkpoint path and tell a model that has converged from one still learning or one
  that has begun to fit its training window.
- See why every checkpoint is published rather than the one whose validation IC is highest.

**Book reference**: Chapter 14, Section 14.6 (The conditional autoencoder). Chapter 6,
Section 6.7 (Search accounting and run logging) introduces the run log this notebook writes into.

**Prerequisites**: [`03_financial_features`](03_financial_features.ipynb),
[`04_evaluation`](04_evaluation.ipynb), [`08_latent_factors`](08_latent_factors.ipynb) and
[`08a_ipca`](08a_ipca.ipynb), which is the linear version of this model.

**What it writes**: one training run per label and one complete validation prediction set per
label and epoch checkpoint, in `run_log/registry.db` and under `run_log/training/` and
`run_log/predictions/`, grouped under a population named for this model. The family splits across
four notebooks, so each publishes its own population rather than one shared one.
[`11_backtest`](11_backtest.ipynb) reads them and selects on validation backtest Sharpe.
**Selection happens there, not here**, and the checkpoint is part of what is selected.

<!-- ML4T_ZH_BEGIN c0000-p001 -->
- 區分潛在因子模型對結構與對函數形式的假設。
- 閱讀檢查點路徑，區分已收斂、仍在學習及已開始過度擬合訓練視窗的模型。
- 了解為何發布每個檢查點，而非只發布驗證資訊係數最高的一個。

**書籍參考**：第 14 章第 14.6 節（條件式自編碼器）。第 6 章第 6.7 節（搜尋計數與執行紀錄）介紹本 Notebook 寫入的執行紀錄。

**先備條件**：[`03_financial_features`](03_financial_features.ipynb)、[`04_evaluation`](04_evaluation.ipynb)、[`08_latent_factors`](08_latent_factors.ipynb)，以及此模型線性版本的 [`08a_ipca`](08a_ipca.ipynb)。

**寫出內容**：每個標籤各有一次訓練執行；每個標籤與訓練週期檢查點各有一組完整驗證預測，儲存於 `run_log/registry.db`、`run_log/training/` 與 `run_log/predictions/` 之下，並歸入以此模型命名的模型群體。家族分散於四份 Notebook，因此各自發布自己的模型群體，而非共用一個。[`11_backtest`](11_backtest.ipynb) 讀取它們，依驗證回測夏普比率選擇。**選擇在那裡進行，而非此處**，檢查點也是選擇內容的一部分。
<!-- ML4T_ZH_END c0000-p001 -->


In [ ]:
"""Fit the declared firm-characteristics conditional-autoencoder population."""

import plotly.graph_objects as go
import polars as pl
import yaml

from case_studies.research import (
    declared_labels,
    load_model_configs,
    model_requests,
    open_study,
    plan_models,
    planned_model_plan,
    primary_label,
    run_model_population,
    supersedes_for_run,
)
from utils.paths import get_case_study_dir
from utils.style import COLORS, show_plotly_with_alt

In [ ]:
LABELS: list[str] = []
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
PREVIEW_REDUCTIONS: dict = {}
POPULATION_NAME = ""
SUPERSEDES_POPULATION: str = ""
# The device to fit on. Empty means the one declared in config/setup.yaml.
DEVICE: str = ""

MODEL_NAME = "cae"

In [ ]:
study = open_study(
    "us_firm_characteristics", execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None
)

## 1. Which labels, and what the configuration says

Every label whose training menu declares `latent_factors:` is fitted, and all three do:
`fwd_ret_1m`, `fwd_ret_1m_win` - the same return with each month's cross-section clipped at its
own tails - and `fwd_class_1m`, which turns that return into a class.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 1. 使用哪些標籤，以及設定說明什麼

凡訓練選單宣告 `latent_factors:` 的標籤都會擬合，三種標籤皆然：`fwd_ret_1m`、`fwd_ret_1m_win`——即在各月橫斷面自身的尾端截尾後的相同報酬——以及將該報酬轉成類別的 `fwd_class_1m`。
<!-- ML4T_ZH_END c0004-p000 -->


In [ ]:
declared_labels(study, "latent_factors")

`case_studies/config/cae/cae.yaml` declares the factor count, the training budget and the
checkpoint interval. The budget and the interval together decide the checkpoint surface: a state
is saved every `checkpoint_interval` epochs up to `n_epochs`, and each saved state becomes a
registered prediction set covering the whole validation period. That is why the plan below shows
ten checkpoints where [`08a_ipca`](08a_ipca.ipynb) shows one. The runtime comes from
`config/setup.yaml` under `modeling.latent_factors` and is CUDA, which this model can use and
IPCA cannot.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
`case_studies/config/cae/cae.yaml` 宣告因子數、訓練預算及檢查點間隔。預算與間隔共同決定檢查點集合：每隔 `checkpoint_interval` 個訓練週期儲存一次狀態，直到 `n_epochs`；每個已儲存狀態都成為涵蓋完整驗證期間的登錄預測集合。因此下方計畫顯示十個檢查點，而 [`08a_ipca`](08a_ipca.ipynb) 只有一個。執行環境來自 `config/setup.yaml`，位於 `modeling.latent_factors` 之下，設定為 CUDA；本模型可使用 CUDA，IPCA 則不能。
<!-- ML4T_ZH_END c0006-p000 -->


In [ ]:
configs = load_model_configs(
    study,
    "latent_factors",
    labels=LABELS or None,
    config_names=[MODEL_NAME],
)
configs

`LABELS` narrows what is fitted, and a narrowed run declares a different set of members than the
canonical population does. A population is immutable once written, so such a run must publish
under its own name: on a fresh workspace it would otherwise register an incomplete snapshot under
the canonical one, and where the full population already exists the registry refuses it.

The comparison is against this model's own declared rows rather than the whole `latent_factors`
catalog. The family is split across four notebooks and each publishes one model, so the complete
catalog is four times what this one publishes, and comparing against it would report every
canonical run as narrowed.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
`LABELS` 會縮小擬合範圍，而縮小後的執行宣告的成員集合與標準模型群體不同。模型群體一旦寫出便不可變動，因此這類執行必須以專屬名稱發布：否則在全新工作區中，會以標準名稱登記不完整快照；完整群體若已存在，登錄系統則會拒絕。

此處比較的是本模型自身宣告的資料列，而非整個 `latent_factors` 目錄。家族分散於四份 Notebook，各自發布一個模型，因此完整目錄的規模是本 Notebook 發布內容的四倍；若對整個目錄比較，每次標準執行都會被誤報為範圍縮小。
<!-- ML4T_ZH_END c0008-p000 -->


In [ ]:
declared = load_model_configs(study, "latent_factors", config_names=[MODEL_NAME])
if set(configs.get_column("label")) != set(declared.get_column("label")) and not POPULATION_NAME:
    raise ValueError(
        f"this run fits {configs.height} of the {declared.height} declared labels, so it cannot "
        "publish the canonical population; pass POPULATION_NAME to give it its own"
    )

The device is the second thing that can put a run outside the declared population, and it is
not a note beside the result: a network trained on a GPU and the same network trained on a CPU
accumulate their sums in different orders and reach different weights, so the two are different
computations and get different identities. `config/setup.yaml` declares the device this family
publishes on under `modeling.latent_factors`, and the runner refuses a device the machine does
not have rather than falling back to one it does. A reader without an NVIDIA card passes
`DEVICE="cpu"` with a `POPULATION_NAME` of their own.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
裝置是使執行可能落在宣告模型群體之外的第二項因素，而且不只是結果旁的註記：以 GPU 訓練的網路與以 CPU 訓練的相同網路，以不同順序累加總和，最終得到不同權重，因此屬於不同計算，具有不同識別碼。`config/setup.yaml` 在 `modeling.latent_factors` 下宣告此家族發布所用的裝置，若機器沒有要求的裝置，執行器會拒絕，而非自行改用其他裝置。沒有 NVIDIA 顯示卡的讀者，可傳入 `DEVICE="cpu"`，並使用自己的 `POPULATION_NAME`。
<!-- ML4T_ZH_END c0010-p000 -->


In [ ]:
setup = yaml.safe_load(
    (get_case_study_dir("us_firm_characteristics") / "config" / "setup.yaml").read_text()
)
PUBLISHED_DEVICE = str(setup["modeling"]["latent_factors"]["device"])
device = DEVICE or PUBLISHED_DEVICE

if device != PUBLISHED_DEVICE and not POPULATION_NAME:
    raise ValueError(
        f"this run fits on device {device!r} rather than the declared {PUBLISHED_DEVICE!r}, so "
        "it cannot publish the canonical population; pass POPULATION_NAME to give it its own"
    )

## 2. Binding the declarations to the data

Planning reads the label and feature files, computes the fold boundaries from the walk-forward
parameters in `config/setup.yaml`, works out the exact rows each fit must predict, and derives
every training and prediction identity - without fitting anything. Four things to check:

- **`feature_count` and `eligible_rows` agree across the rows.** A row that differs is a label
  measured on a different sample from its neighbours.
- **`folds` is the same everywhere**, and equals the number of walk-forward splits
  [`04_evaluation`](04_evaluation.ipynb) established.
- **`validation_start` and `validation_end` bracket the development sample**, with none of the
  held-out 2016 tail visible.
- **`checkpoints` is how many training states each label will publish predictions for.**
  Multiply it by the number of rows to get the number of candidate models this notebook creates.

<!-- ML4T_ZH_BEGIN c0012-p000 -->
## 2. 將宣告與資料連結

規劃過程讀取標籤與特徵檔，依 `config/setup.yaml` 中的前推參數計算各折邊界，確定每次擬合須預測的精確資料列，並推導所有訓練與預測識別碼；全程不擬合模型。需檢查四件事：

- **`feature_count` 與 `eligible_rows` 在各列一致。** 不同的列表示某標籤是在與其他標籤不同的樣本上衡量。
- **`folds` 在各處相同**，且等於 [`04_evaluation`](04_evaluation.ipynb) 確立的前推切分數。
- **`validation_start` 與 `validation_end` 涵蓋開發樣本首尾**，不顯示任何 2016 年保留測試期間尾段。
- **`checkpoints` 是每個標籤將發布預測的訓練狀態數。** 將其乘以資料列數，即得本 Notebook 建立的候選模型數。
<!-- ML4T_ZH_END c0012-p000 -->


In [ ]:
requests = model_requests(
    study,
    configs,
    execution_tier=EXECUTION_TIER,
    overrides={"device": device},
    preview_reductions=PREVIEW_REDUCTIONS,
    notebook="08b_conditional_autoencoder",
)
plan = plan_models(study, requests=requests)

planned_model_plan(plan).select(
    "label",
    "config_name",
    "task",
    "feature_count",
    "eligible_rows",
    "folds",
    "checkpoints",
    "validation_start",
    "validation_end",
)

The plan is also where a short population is visible. A run that fitted fewer labels than the
menu declares, or published fewer checkpoints than the configuration does, would still print an
IC table and still register its rows; what it would not do is announce the gap.

<!-- ML4T_ZH_BEGIN c0014-p000 -->
計畫也是發現模型群體成員不足的地方。若執行擬合的標籤少於選單宣告數，或發布的檢查點少於設定宣告數，仍會列印資訊係數表並登錄資料列，卻不會指出缺口。
<!-- ML4T_ZH_END c0014-p000 -->


In [ ]:
planned_pairs = {(member.label, member.config_name) for member in plan.members}
requested_pairs = set(zip(configs["label"], configs["config_name"], strict=True))
if planned_pairs != requested_pairs:
    raise RuntimeError(
        "the plan does not match the loaded conditional-autoencoder menu; "
        f"missing {sorted(requested_pairs - planned_pairs)}, "
        f"unexpected {sorted(planned_pairs - requested_pairs)}"
    )
print(f"{len(requested_pairs)} label-configuration pairs")
print(f"{len(plan.expected_training_hashes)} training identities")
print(f"{len(plan.expected_prediction_hashes)} validation prediction sets")

## 3. Fitting the population

`run_model_population` runs every planned request. For one request it walks the folds, and on
each one:

1. takes the firm-months inside that fold's training window,
2. trains the network that maps a firm's characteristics to its exposures jointly with the
   linear part that turns each month's returns into factor returns, for the declared number of
   epochs, saving the fitted state at each checkpoint on the way,
3. applies each saved state to the validation months' characteristics to get exposures there,
   and multiplies by the factor returns to get a predicted return.

Step 3 is what makes one fit produce many results. Fold predictions are concatenated into one
series per checkpoint covering the whole validation period, and each becomes its own registered
prediction set with its own identity. Nothing here chooses among them.

**This notebook passes the plan rather than resolved requests, and on this family that changes
what the plan table could show rather than how the fitting is ordered.** The linear, boosted and
TabM adapters each provide a batch runner that prepares one fold set for several configurations;
the latent-factor adapter provides none, so every request is fitted on its own whichever path is
taken. There is nothing here for a batch runner to share in any case: this notebook publishes one
model against three labels, and each label is a different set of rows. What passing the plan does
cost is `eligible_entities`, which needs the eligibility keys themselves; `eligible_rows` above
moves whenever the universe does, so the check that column existed for is still made.


<!-- ML4T_ZH_BEGIN c0016-p000 -->
## 3. 擬合模型群體

`run_model_population` 執行每個已規劃的請求。對單一請求，它逐折處理，每折依序：

1. 取出該折訓練視窗內的公司月份；
2. 同時訓練將公司特徵映射至曝險的網路，以及把各月報酬轉換為因子報酬的線性部分，訓練宣告的週期數，途中在每個檢查點儲存擬合狀態；
3. 將各個已儲存狀態套用至驗證月份的特徵，取得該期間的曝險，再乘以因子報酬得到預測報酬。

第 3 步使一次擬合產生多項結果。各折預測依檢查點串接成涵蓋完整驗證期間的序列，每個序列都成為具有自身識別碼的獨立登錄預測集合。此處不在它們之間作選擇。

**本 Notebook 傳入計畫，而非已解析的請求；在此模型家族中，這會改變計畫表可顯示的內容，卻不改變擬合順序。** 線性、提升及 TabM 轉接器各有批次執行器，可為多項設定準備一組折；潛在因子轉接器則沒有，因此無論採哪條路徑，每個請求都單獨擬合。此處也沒有可供批次執行器共用的內容：本 Notebook 對三種標籤發布一個模型，而每種標籤的資料列集合不同。傳入計畫所付出的代價是無法顯示需要資格鍵值本身的 `eligible_entities`；上方的 `eligible_rows` 會隨投資範圍變動，所以該欄原本要進行的檢查仍會執行。
<!-- ML4T_ZH_END c0016-p000 -->

**What the call publishes is a population**: a named, immutable list of the prediction sets it
will produce, written down before the first fit. Afterwards every member must exist and be
complete, which is what makes the downstream comparison well defined.

<!-- ML4T_ZH_BEGIN c0016-p001 -->
**呼叫會發布一個模型群體**：在首次擬合前寫下的具名、不可變動預測集合清單。之後每個成員都必須存在且完整，下游比較才有明確定義。
<!-- ML4T_ZH_END c0016-p001 -->


In [ ]:
population_name = POPULATION_NAME or f"us_firm_characteristics-{MODEL_NAME}-validation-v1"
# The declared hash is only meaningful where a generation of this name already exists. A preview
# run, a reader's first canonical run against an empty `run_log/`, and a run under a caller-chosen
# `POPULATION_NAME` are all refused by `OfficialPopulation.create` if it is passed anyway. The
# resolution lives in shared code so no notebook branches on the tier.
supersedes = supersedes_for_run(
    study,
    population_name=population_name,
    declared=SUPERSEDES_POPULATION,
    execution_tier=EXECUTION_TIER,
)
execution, population = run_model_population(
    study, plan, population_name=population_name, supersedes=supersedes
)

print(f"{len(execution.runs)} configurations, fitted or served from the registry")
print(f"population {population.name}: {len(population.members)} prediction sets")

Re-running this notebook unchanged costs the time it takes to read the data. Every identity is
re-derived from the inputs, the registry already holds the matching rows, and the runner returns
the stored result rather than fitting again.

**There are no fold counts above, and their absence is the honest reading rather than an
omission.** [`05_linear`](05_linear.ipynb) and [`06_gbm`](06_gbm.ipynb) print folds fitted
against folds served from the registry, because the linear and GBM runners record
`fitted_folds` and `reused_folds` on every path they take. The latent-factor runner records
neither - its result carries no diagnostics at all, so the only per-run keys are the status and
the training hash. Printing two zeros here would say every fold came from cache, which is the
opposite of what nothing-recorded means.

[`07_tabular_dl`](07_tabular_dl.ipynb) sits between the two and says so at run time. Its runner
records fold counts when it fits and none when it serves a whole configuration from the
registry, so on a re-run it reports how many of its runs recorded counts instead of printing a
split it was never given.

### Running configurations of your own

The published run log is read-only. To add runs, open the study against a workspace, which holds
its own registry and artifacts and reads the same labels and features:

```python
study = open_study("us_firm_characteristics", workspace="~/ml4t-experiments")
configs = load_model_configs(study, "latent_factors", labels=["fwd_ret_1m"], config_names=["cae"])
requests = model_requests(study, configs)
plan = plan_models(study, requests=requests)
execution, population = run_model_population(study, plan, population_name="my-cae-v1")
```


<!-- ML4T_ZH_BEGIN c0018-p000 -->
不修改本 Notebook 直接重跑，只需讀取資料的時間。系統依輸入重新推導各識別碼，登錄系統已有相符資料列，執行器便回傳儲存結果而不重新擬合。

**上方沒有各折計數，這是忠實呈現，而非遺漏。** [`05_linear`](05_linear.ipynb) 與 [`06_gbm`](06_gbm.ipynb) 會印出實際擬合與由登錄系統提供的折數，因為線性及 GBM 執行器在每條路徑都記錄 `fitted_folds` 與 `reused_folds`。潛在因子執行器兩者都不記錄；其結果完全沒有診斷資料，所以每次執行只有狀態與訓練雜湊兩項鍵值。若在此印出兩個零，就會表示每折都來自快取，這與「未記錄」的真意相反。

[`07_tabular_dl`](07_tabular_dl.ipynb) 介於兩者之間，並在執行時說明。其執行器在擬合時記錄各折計數，但由登錄系統提供完整設定時不記錄，因此重跑時會報告有多少次執行記錄了計數，而非印出從未取得的劃分。

### 執行自己的設定

已發布的執行紀錄為唯讀。若要新增執行，請在工作區中開啟研究；工作區有自己的登錄系統與產出檔案，並讀取相同標籤與特徵：

```python
study = open_study("us_firm_characteristics", workspace="~/ml4t-experiments")
configs = load_model_configs(study, "latent_factors", labels=["fwd_ret_1m"], config_names=["cae"])
requests = model_requests(study, configs)
plan = plan_models(study, requests=requests)
execution, population = run_model_population(study, plan, population_name="my-cae-v1")
```
<!-- ML4T_ZH_END c0018-p000 -->

To change the factor count, the training budget or the checkpoint interval, edit
`case_studies/config/cae/cae.yaml`. Each changes this configuration's identity - the interval as
much as the budget, because it changes which states exist - so the result registers as a new row
beside the old one rather than replacing it.
[`RUN_LOG.md`](../RUN_LOG.md#running-your-own-configurations) covers the rest.

<!-- ML4T_ZH_BEGIN c0018-p001 -->
若要改變因子數、訓練預算或檢查點間隔，請編輯 `case_studies/config/cae/cae.yaml`。每一項變更都會改變此設定的識別碼；間隔與預算同樣重要，因為它會改變存在哪些狀態。因此新結果會在舊結果旁另登一列，而非取代。其餘細節請參閱 [`RUN_LOG.md`](../RUN_LOG.md#running-your-own-configurations)。
<!-- ML4T_ZH_END c0018-p001 -->


## 4. What came out

One row per label and epoch checkpoint. `ic_mean` is the **information coefficient**: in each
validation month, rank the firms by the model's prediction, rank them by the return they went on
to earn, correlate the two rankings, and average that monthly correlation over the validation
period.

**Every count and aggregate below is keyed on `(label, checkpoint_value)`.** Grouping on the
checkpoint alone would average across targets; grouping on the label alone would collapse the
training path this notebook exists to show.

The published catalog is checked against the population planned before fitting rather than
against its own row count, because a run that lost a member would otherwise report a shorter
table and nothing else.

<!-- ML4T_ZH_BEGIN c0019-p000 -->
## 4. 產出結果

每個標籤與訓練週期檢查點各占一列。`ic_mean` 是**資訊係數**：在各驗證月份，依模型預測為公司排名，再依其後取得的報酬排名，計算兩組排名的相關性，最後對驗證期間的月度相關性取平均。

**下方所有計數與彙總都以 `(label, checkpoint_value)` 為鍵。** 若只依檢查點分組，會對不同目標取平均；若只依標籤分組，會將本 Notebook 原本要呈現的訓練路徑合併掉。

已發布的目錄會與擬合前規劃的模型群體比較，而非只檢查目錄自身的資料列數；否則少了一個成員的執行只會報告較短的表格，沒有其他提示。
<!-- ML4T_ZH_END c0019-p000 -->


In [ ]:
catalog = execution.catalog_rows.select(
    "config_name",
    "label",
    "task",
    "complete",
    "checkpoint_kind",
    "checkpoint_value",
    "ic_mean",
    "ic_std",
    "ic_t",
    "ic_n_days",
    "auc_mean_daily",
    pl.col("direction_label").alias("auc_scored_against"),
    "n_folds",
    "training_hash",
    "prediction_hash",
).sort(["label", "checkpoint_value"])

if set(catalog.get_column("prediction_hash")) != set(plan.expected_prediction_hashes):
    raise RuntimeError("the published catalog differs from the population planned before fitting")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("a partial conditional-autoencoder checkpoint cannot pass to backtesting")
if catalog.select("label", "config_name", "checkpoint_value").n_unique() != catalog.height:
    raise RuntimeError("each label and checkpoint must identify one prediction set")
if catalog.get_column("checkpoint_value").null_count():
    raise RuntimeError("every checkpoint must name the epoch it was taken at")

catalog = catalog.with_columns(
    full_coverage=pl.col("ic_n_days") == pl.col("ic_n_days").max().over("label")
)

primary = primary_label(study)
present = sorted(set(catalog.get_column("label")))
# The primary label leads when it was fitted. A subset run that leaves it out orders the panels by
# whichever label it did fit rather than by one that is not there.
panel_labels = [label for label in [primary] if label in present] + [
    label for label in present if label != primary
]
print(f"{catalog.height} candidate models across {len(panel_labels)} labels")
print(f"at {catalog.n_unique('checkpoint_value')} checkpoints each")
catalog.select(
    "label",
    "checkpoint_value",
    "ic_mean",
    "ic_std",
    "ic_t",
    "ic_n_days",
    "full_coverage",
).head(15)

### What more training does

Each line traces one label's out-of-sample IC as epochs are added, and the shape to read is where
it reaches its highest point.

A line that rises and then flattens has learned what it is going to learn. A line still climbing
at the last checkpoint has not converged at the declared budget, so its final number is a lower
bound rather than a level. A line that peaks early and then falls is the third case: added epochs
stop buying out-of-sample ranking and start costing it. All three are published, because which
one a reader is looking at is a fact about the data rather than something to resolve away before
the backtest sees it.

What the chart cannot say is why. Only validation IC is computed here; the reconstruction loss on
the training window is not, so a falling line is evidence that more training stopped helping and
not evidence about what the network was doing to its inputs while that happened.

<!-- ML4T_ZH_BEGIN c0021-p000 -->
### 增加訓練量會發生什麼事

每條線追蹤一種標籤隨訓練週期增加的樣本外資訊係數，閱讀時應注意它在哪裡達到最高點。

先上升後趨於平坦的線，表示模型已學到目前所能學的內容。若最後一個檢查點仍持續上升，表示在宣告的預算下尚未收斂，其最終數字是下限，而非穩定水準。早期達峰後下降則是第三種情況：增加訓練週期不再改善樣本外排名，反而開始傷害它。三種情況都會發布，因為讀者面對哪一種，是資料的事實，而非在回測前應消除的細節。

圖表無法說明原因。此處只計算驗證資訊係數，沒有計算訓練視窗的重建損失，所以曲線下降只能證明更多訓練不再有幫助，無法證明當時網路如何處理輸入。
<!-- ML4T_ZH_END c0021-p000 -->


In [ ]:
curves = catalog.filter("full_coverage").sort("label", "checkpoint_value")
fig = go.Figure()
for label in panel_labels:
    series = curves.filter(pl.col("label") == label).sort("checkpoint_value")
    fig.add_trace(
        go.Scatter(
            x=series.get_column("checkpoint_value").to_list(),
            y=series.get_column("ic_mean").to_list(),
            mode="lines+markers",
            name=f"{label} ({'primary' if label == primary else 'variant'})",
            line=dict(
                color=COLORS["blue"] if label == primary else COLORS["slate"],
                width=2 if label == primary else 1.5,
            ),
            marker=dict(size=6),
        )
    )
fig.add_hline(y=0, line_width=1, line_dash="dash", line_color=COLORS["neutral"])
fig.update_xaxes(title_text="Training epochs completed")
fig.update_yaxes(title_text="Mean IC (validation)")
fig.update_layout(
    title="More training stops helping the out-of-sample ranking",
    height=460,
    width=900,
    margin=dict(t=90),
    legend=dict(title_text="Label"),
)
# Where each line peaks is a fact about the frame, so the alt text counts it rather than asserting
# a shape the next run may not reproduce.
peaks = (
    curves.group_by("label")
    .agg(
        peak_checkpoint=pl.col("checkpoint_value").sort_by("ic_mean", descending=True).first(),
        first_checkpoint=pl.col("checkpoint_value").min(),
        last_checkpoint=pl.col("checkpoint_value").max(),
    )
    .sort("label")
)
peak_text = "; ".join(
    f"{row['label']} peaks at epoch {row['peak_checkpoint']} of "
    f"{row['first_checkpoint']} to {row['last_checkpoint']}"
    for row in peaks.iter_rows(named=True)
)
show_plotly_with_alt(
    fig,
    "Line chart of mean validation information coefficient against training epochs completed, one "
    "line per label with a marker at each declared checkpoint: the primary target in dark navy and "
    "the two variants in slate, over a dashed zero line. Counted from the frame: "
    f"{peak_text}.",
)

### How much the stopping point is worth

The frame below puts the range a label's IC covers across its own checkpoints against the spread
across labels at a fixed amount of training. That is the quantity deciding whether the stopping
point is a decision worth making carefully or one being made by noise, and both sides are
computed within this model so the comparison is not smuggling in a difference between models.

<!-- ML4T_ZH_BEGIN c0023-p000 -->
### 停止時點有多少價值

下方資料框將某標籤自身檢查點之間的資訊係數範圍，與固定訓練量下各標籤之間的差距相比。這個量決定停止時點是值得謹慎選擇的決策，還是主要受雜訊影響；兩側都在此模型內計算，所以比較不會悄悄混入模型間的差異。
<!-- ML4T_ZH_END c0023-p000 -->


In [ ]:
final_epoch = int(curves.get_column("checkpoint_value").max())
checkpoint_vs_label = (
    curves.group_by("label")
    .agg(
        ic_min=pl.col("ic_mean").min(),
        ic_max=pl.col("ic_mean").max(),
        ic_final=pl.col("ic_mean").filter(pl.col("checkpoint_value") == final_epoch).first(),
        scored_months=pl.col("ic_n_days").max(),
    )
    .with_columns(checkpoint_range=pl.col("ic_max") - pl.col("ic_min"))
    .sort("label")
)
across_labels = float(
    checkpoint_vs_label.get_column("ic_final").max()
    - checkpoint_vs_label.get_column("ic_final").min()
)
print(f"compared at {final_epoch} epochs; spread across labels there: {across_labels:.4f}")
checkpoint_vs_label

## 5. What to notice

**This model and [`08a_ipca`](08a_ipca.ipynb) differ in one function and nothing else.** Same
characteristics, same folds, same months scored, same factor count, same conditional-factor
structure - a firm's exposure comes from its own characteristics, and the predicted return is
that exposure times the month's factor return. What changes is whether the map from
characteristics to exposures is a matrix or a network. So a gap between the two notebooks is
evidence about functional form, and it is one of the few comparisons in this case study clean
enough to read that way.

**A relaxed constraint is not free.** The linear map had few enough parameters that the whole
panel could pin them down; a network has enough that it can fit the training window's noise as
well as its structure, and the checkpoint path is where that becomes visible. This is the same
tension [`07_tabular_dl`](07_tabular_dl.ipynb) shows under a different architecture, and it is
why both notebooks publish every checkpoint rather than a chosen one.

**The checkpoint is part of the configuration, and that is a deliberate cost.** Publishing ten
states per label makes the downstream backtest ten times larger than it would be if this notebook
picked one. Picking one would mean picking it on validation IC, and IC is not what the strategy
is selected on - so the choice would be made by the wrong quantity, and made somewhere the reader
cannot see it. `checkpoint_vs_label` is how to judge whether that cost is buying anything on this
data: where the within-label range across checkpoints exceeds the spread across labels at a fixed
epoch, the stopping point matters more than the target does.


<!-- ML4T_ZH_BEGIN c0025-p000 -->
## 5. 值得注意的結果

**本模型與 [`08a_ipca`](08a_ipca.ipynb) 只在一個函數上不同。** 特徵、各折、評分月份、因子數及條件因子結構都相同：公司曝險來自自身特徵，預測報酬是該曝險乘以當月因子報酬。改變的只有特徵到曝險的映射是矩陣還是網路。因此，兩份 Notebook 的差距是函數形式的證據，也是本案例研究少數足以如此清楚解讀的比較之一。

**放寬限制有其代價。** 線性映射的參數少，整份縱橫資料足以將其估計清楚；網路的參數多，除了結構也可能擬合訓練視窗的雜訊，而檢查點路徑會顯示這種情況。[`07_tabular_dl`](07_tabular_dl.ipynb) 在不同架構下呈現相同取捨，這也是兩份 Notebook 都發布每個檢查點，而非只發布選定的一個的原因。

**檢查點是設定的一部分，且這是刻意承擔的代價。** 每個標籤發布十個狀態，使下游回測的規模是本 Notebook 自行選一個狀態的十倍。若在此選一個，就意味著依驗證資訊係數選擇，但策略並非依資訊係數選擇；如此便會用錯誤的量，在讀者看不見的地方作決策。`checkpoint_vs_label` 可用來判斷這項成本在此資料上是否有價值：若單一標籤跨檢查點的範圍超過固定訓練週期下各標籤的差距，停止時點就比目標更重要。
<!-- ML4T_ZH_END c0025-p000 -->

**None of this selects anything.** IC measures whether predictions rank firms correctly, not
whether a strategy trading them makes money after costs and turnover. Selection is on validation
backtest Sharpe in [`11_backtest`](11_backtest.ipynb), where the checkpoint is part of what is
selected.

**Known limitations.** The factor count, the training budget, the checkpoint interval and the
network's width are declared rather than chosen, and nothing here searches over them - a search
over validation IC is what this arrangement exists to avoid. The IC is an average of monthly rank
correlations with no adjustment for serial dependence, so `ic_t` is a diagnostic rather than a
test. And every number is measured on validation folds that have been read many times over by the
time a case study reaches this notebook.

<!-- ML4T_ZH_BEGIN c0025-p001 -->
**上述結果都不作出選擇。** 資訊係數衡量預測是否正確排序公司，而非交易策略扣除成本及週轉後能否獲利。選擇依據是 [`11_backtest`](11_backtest.ipynb) 中的驗證回測夏普比率，檢查點也是選擇內容的一部分。

**已知限制。** 因子數、訓練預算、檢查點間隔及網路寬度都是宣告值，而非經選擇所得；此處不對它們搜尋，因為整體安排正是為了避免依驗證資訊係數搜尋。資訊係數是月度排名相關性的平均值，未調整序列相依性，因此 `ic_t` 是診斷，而非檢驗。此處每個數字都在驗證折上衡量，而案例研究進行至本 Notebook 時，這些驗證折已被反覆查看多次。
<!-- ML4T_ZH_END c0025-p001 -->


**Next**: [`08c_stochastic_discount_factor`](08c_stochastic_discount_factor.ipynb) drops the
two-stage structure both of these share. Instead of estimating a factor representation and then
forecasting it, it learns the pricing object directly from a no-arbitrage condition - so there is
no factor-return history to hand to a second stage at all.

<!-- ML4T_ZH_BEGIN c0026-p000 -->
**接下來**：[`08c_stochastic_discount_factor`](08c_stochastic_discount_factor.ipynb) 會捨棄兩者共用的兩階段結構。它不先估計因子表示再進行預測，而是從無套利條件直接學習定價對象，因此沒有因子報酬歷史可交給第二階段。
<!-- ML4T_ZH_END c0026-p000 -->
